# Q7: SHORT versions you can type by hand (no copy-paste)
If the exam room does not allow copy-paste, `auto_prepare` (Q5) is too long to type.
These 2 short functions do the important parts, about **40 lines** in total:

| function | lines | does |
|---|---|---|
| `clean(df, target)` | ~22 | drop duplicates, "?"/-999 → missing, "$1,234" → number, text target → 0/1, fill missing, drop name-like text, one-hot |
| `best_degree(x, y)` | ~18 | 5-fold CV for degree 1-6, picks the smallest degree within 0.01 of the best |

Use:
```python
df = clean(pd.read_csv("../exam.csv"), target="y")
x, y = df[cols].values, df["y"].values
print("degree", best_degree(x, y))
```
**Tip:** before the exam, practise typing them once. If copy-paste IS allowed, use Q5 instead (it does more).

## 1. clean (type this)

In [1]:
import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype as isnum

def clean(df, target):
    df = df.drop_duplicates().replace(["?", "NA", "-", "", -999], np.nan)
    df = df.dropna(subset=[target])
    for c in df.columns:
        if not isnum(df[c]):
            num = pd.to_numeric(df[c].astype(str).str.replace(r"[$,% ]", "", regex=True), errors="coerce")
            if num.notna().mean() > 0.8:
                df[c] = num
    if not isnum(df[target]):
        df[target] = df[target].astype("category").cat.codes
    for c in df.columns.drop(target):
        if not isnum(df[c]):
            df[c] = df[c].str.strip().str.lower()
            if df[c].nunique() > 20:
                df = df.drop(columns=c)
                continue
            df[c] = df[c].fillna(df[c].mode()[0])
        else:
            df[c] = df[c].fillna(df[c].median())
    return pd.get_dummies(df, drop_first=True, dtype=int)

## 2. best_degree (type this, Q1 only: numpy from scratch)

In [2]:
def best_degree(x, y, max_d=6):
    idx = np.random.default_rng(42).permutation(len(y))
    folds = np.array_split(idx, 5)
    scores = []
    for d in range(1, max_d + 1):
        X = np.column_stack([x ** k for k in range(1, d + 1)])
        X = (X - X.mean(0)) / (X.std(0) + 1e-9)
        X = np.c_[np.ones(len(y)), X]
        s = []
        for f in folds:
            tr = np.setdiff1d(idx, f)
            w = np.linalg.pinv(X[tr]) @ y[tr]
            p = X[f] @ w
            s.append(1 - ((y[f] - p) ** 2).sum() / ((y[f] - y[f].mean()) ** 2).sum())
        scores.append(np.mean(s))
    print("CV R2 per degree:", np.round(scores, 4))
    return int(np.argmax(np.array(scores) >= max(scores) - 0.01)) + 1

## 3. Test on titanic: same result as the long version?

In [3]:
df = clean(pd.read_csv("../titanic.csv"), target="Fare")
print(df.shape, list(df.columns))
cols = ["Age", "Pclass", "SibSp"]
x, y = df[cols].values.astype(float), df["Fare"].values
print("best degree:", best_degree(x, y))

(891, 10) ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Sex_male', 'Embarked_q', 'Embarked_s']
CV R2 per degree: [0.37   0.4217 0.4199 0.4196 0.4105 0.4166]
best degree: 2


## 4. Test on dirty data

In [4]:
rng = np.random.default_rng(0)
size = rng.uniform(30, 200, 60).round(1)
dirty = pd.DataFrame({
    "Size": size,
    "City": rng.choice([" BKK", "bkk ", "Cnx", None], 60),
    "Price": [f"${p:,.0f}" for p in 30 * size + 0.4 * size ** 2 + rng.normal(0, 300, 60)],
})
dirty.loc[3, "Size"] = -999
dirty.loc[7, "Price"] = "?"
dirty = pd.concat([dirty, dirty.head(3)])
print(dirty.head(8))

df = clean(dirty, target="Price")
print(df.head(), df.shape, "| missing left:", df.isna().sum().sum())
print("best degree:", best_degree(df[["Size"]].values, df["Price"].values), "(true = 2)")

    Size  City    Price
0  138.3   Cnx  $11,573
1   75.9  bkk    $5,088
2   37.0   NaN   $1,571
3 -999.0   BKK   $1,887
4  168.3   NaN  $16,249
5  185.2   BKK  $19,055
6  133.1   Cnx  $11,154
7  154.0   Cnx        ?
    Size  Price  City_cnx
0  138.3  11573         1
1   75.9   5088         0
2   37.0   1571         0
3  122.2   1887         0
4  168.3  16249         0 (59, 3) | missing left: 0
CV R2 per degree: [0.9427 0.9705 0.9705 0.9693 0.969  0.9677]
best degree: 2 (true = 2)


## 5. Q2 (sklearn allowed) = even shorter: let sklearn find the degree
```python
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
g = GridSearchCV(make_pipeline(PolynomialFeatures(), StandardScaler(), LinearRegression()),
                 {"polynomialfeatures__degree": range(1, 7)}, cv=5).fit(x_train, y_train)
print(g.best_params_)
```

In [5]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
g = GridSearchCV(make_pipeline(PolynomialFeatures(), StandardScaler(), LinearRegression()),
                 {"polynomialfeatures__degree": range(1, 7)}, cv=5).fit(df[["Size"]].values, df["Price"].values)
print(g.best_params_)

{'polynomialfeatures__degree': 2}
